In [1]:
import gymnasium as gym
import numpy as np
import math
import os
import configparser
from sb3_contrib.common.maskable.policies import MaskableActorCriticPolicy
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.ppo_mask import MaskablePPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
import matplotlib.pyplot as plt
from sb3_contrib.common.maskable.utils import get_action_masks



In [2]:
from src.hpc_env import HPCenv
from src.validation import Validation
from src.training import Train
from src.baseline import MedianBaseline
from src.utils import mask_fn, get_config_as_dict
from src.carbon_intensity import CarbonIntensity

In [3]:
WORKLOAD_PATH = "data/workloads/lublin_256.swf"

# Load config with explicit path and typed parsing
config = configparser.ConfigParser()
config_path = os.path.join(os.getcwd(), 'config_file', 'config.ini')
config.read(config_path)

['/Users/mikkeldahl/green_scheduler_v2/config_file/config.ini']

## Model validation

In [4]:
val = Validation()

In [5]:
#checkpoints = [f"seed_0_{step}_steps" for step in 
#               list(range(500_000, 10_000_001, 1_000_000)) + list(range(10_000_000, 50_000_001, 10_000_000))] 
checkpoints = ["seed_1_10000000_steps"]
val.load_dir("results/CI_B8192_RC_LR-00003_ETA5.0_C-None_Lu")

stats = val.validate_policy(
    n_eval_episodes=1,
    debug=True,
    checkpoints=checkpoints,
    mode="validation")


Validating policy on data from:  validation
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
Initating checkpoint:  seed_1_10000000_steps
Val episode:  0
Average carbon next 24 hours:  0.8492313
Current CI:  0.5593185
Delay fixed amount:  86400
current timestamp:  0
Job left:  5643
queue,  [1]
Nodes free,  256
Average carbon next 24 hours:  0.42621288
Current CI:  1.0705044
Job scheduled:  40
Current timestamp (env):  86400
Average carbon next 24 hours:  0.42621288
Current CI:  1.0705044
Delay to finished job: 
Current timestamp (env):  86400
Average carbon next 24 hours:  0.40539297
Current CI:  1.0881329
Job scheduled:  41
Current timestamp (env):  87585
Average carbon next 24 hours:  0.40539297
Current CI:  1.0881329
Delay to finished job: 
Current timestamp (env):  87585
Average carbon next 24 hours:  0.34095323
Current CI:  1.0997578
Delay to finished job: 
Current timestamp

KeyboardInterrupt: 

In [ ]:
stats

4413746.8806942105
4964718.136264815

{'seed_1_10000000_steps': {'Avg Wait': np.float64(6356235.233563707),
  'Max Wait': np.int64(15386461),
  'Avg Response': np.float64(6359322.9705830235),
  'Avg Slowdown': np.float64(802731.354862481),
  'Carbon Emissions': np.float64(4413746.8806942105),
  'Weighted Carbon Emissions': np.float64(-4413746.8806942105),
  'System Utilization': np.float64(0.05518604216707815),
  'Action Analysis': {'Total Actions': 6050,
   'Schedule Action Percentage': 93.27272727272728,
   'Fixed Delay Percentage': 6.545454545454546,
   'Wait Delay Percentage': 0.18181818181818182,
   'Fixed Delays': {'300s': 210, '3600s': 9, '86400s': 177},
   'Wait for Jobs': {'1 jobs': 1, '2 jobs': 0, '3 jobs': 10}}}}

In [ ]:
val.run_baselines(n_eval_episodes=1, mode="validation")

run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
running baseline:  Median Baseline
running baseline:  FCFS Baseline


{'Median Baseline': {'Avg Wait': np.float64(21676.28194222931),
  'Max Wait': np.int64(168198),
  'Avg Response': np.float64(24764.018961545276),
  'Avg Slowdown': np.float64(2556.3680643411494),
  'Carbon Emissions': np.float64(4964718.136264815),
  'Weighted Carbon Emissions': np.float64(-4964718.136264815),
  'System Utilization': np.float64(0.31799142668153796),
  'Action Analysis': {'Total Actions': 14819,
   'Schedule Action Percentage': 38.0794925433565,
   'Fixed Delay Percentage': 29.752344962548083,
   'Wait Delay Percentage': 32.16816249409542,
   'Fixed Delays': {'300s': 4409, '3600s': 0, '86400s': 0},
   'Wait for Jobs': {'1 jobs': 4767, '2 jobs': 0, '3 jobs': 0}}},
 'FCFS Baseline': {'Avg Wait': np.float64(1591.1293638135744),
  'Max Wait': np.int64(90796),
  'Avg Response': np.float64(4678.866383129541),
  'Avg Slowdown': np.float64(169.12755029093884),
  'Carbon Emissions': np.float64(5047757.235819711),
  'Weighted Carbon Emissions': np.float64(-5047757.235819711),
  '

In [ ]:
# Trace replay example: collect action traces, render frames, compile video
from src.validation import Validation
import os

# 1) Point to a trained run directory (contains config.json and logs/)
model_dir = 'results/CI_B8192_RC_LR-00003_ETA5.0_C-None_Lu'  # TODO: set to your run

val = Validation()
val.load_dir(model_dir)
mode = 'validation'  # or 'test'

# 2) Pick a checkpoint from logs/ (choose last by default)
logs_dir = os.path.join(model_dir, 'logs')
available = sorted(os.listdir(logs_dir)) if os.path.isdir(logs_dir) else []
print('Available checkpoints (first 5 shown):', available[5], '... total', len(available))
assert len(available) > 0, 'No checkpoints found in logs/'
checkpoint = available[-1]
print('Using checkpoint:', checkpoint)

# 3) Collect traces for one episode
episodes = val.collect_traces(n_eval_episodes=1, checkpoint=checkpoint, mode=mode, debug=False)
ep = episodes[0]

# 4) Render static 4-line timeseries overview (CI, used procs, avg wait, arrivals)
png_path = val.render_timeseries_plot(
    job_scheduled_history=ep['job_scheduled_history'],
    name=f"timeseries_{checkpoint.replace('.', '_')}_seed{ep['seed']}",
    output_dir='renderings',
    mode=mode,
)
print('Saved timeseries plot at:', png_path)
""" 
# 5) Render a frame-by-frame 4-line timeseries replay and compile to MP4
video_name = f"timeseries_{checkpoint.replace('.', '_')}_seed{ep['seed']}"
mp4_path = val.render_timeseries_video(
    job_scheduled_history=ep['job_scheduled_history'],
    action_trace=ep['action_trace'],
    name=video_name,
    output_dir='renderings',
    mode=mode,
    fps=2,
)
print('Rendered video at:', mp4_path)
 """

Available checkpoints (first 5 shown): seed_0_3500000_steps.zip ... total 30
Using checkpoint: seed_1_9500000_steps.zip
run time mean:  3087.737019315967
run time std:  12205.009945609008
Max Allocated Processors: 128 ;max node: 256 ;max procs: 256 ;max execution time: 160745
Saved timeseries plot at: renderings/timeseries_seed_1_9500000_steps_zip_seed0.png


' \n# 5) Render a frame-by-frame 4-line timeseries replay and compile to MP4\nvideo_name = f"timeseries_{checkpoint.replace(\'.\', \'_\')}_seed{ep[\'seed\']}"\nmp4_path = val.render_timeseries_video(\n    job_scheduled_history=ep[\'job_scheduled_history\'],\n    action_trace=ep[\'action_trace\'],\n    name=video_name,\n    output_dir=\'renderings\',\n    mode=mode,\n    fps=2,\n)\nprint(\'Rendered video at:\', mp4_path)\n '